# Starter Notebook: Transformation And Candidate Comparison

This notebook uses AceBet's **100-row sample only**. The sample demonstrates normalization, derived features, the shared temporal holdout, and in-memory candidate comparison. It cannot reproduce the committed 44,617-row production dataset and is not evidence of production calibration.

The notebook never exports or promotes a model and never writes player-state artifacts.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the AceBet repository root.")


repo_root = find_repo_root(Path.cwd())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
repo_root

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from acebet.dataprep.dataprep import prepare_data
from acebet.features import MODEL_FEATURE_COLUMNS, select_model_features
from acebet.model_artifacts import candidates_with_selection_details, compute_selection
from acebet.train.train import (
    EloBaselineClassifier,
    PreparedTrainingData,
    build_model,
    date_grouped_temporal_holdout,
    evaluate_model,
    evaluate_probabilities,
    get_lgb_params,
)

sample_path = repo_root / "src" / "acebet" / "data" / "atp_data_sample.feather"

## Use The Code-Owned Preparation Boundary

The packaged Feather sample has 100 rows derived from the committed raw sample. prepare_data(...) validates it, normalizes dates, sorts stably, and re-derives deterministic fields. The production training boundary is a different 44,617-row Feather file whose upstream source, licensing, Elo generation, and point-in-time lineage are not recorded here.

In [ ]:
prepared = prepare_data(sample_path)
print(f"Transformation sample rows: {len(prepared)}")
print(f"Date range: {prepared['date'].min().date()} -> {prepared['date'].max().date()}")
prepared[["date", "p1", "p2", "proba_elo", "rank_diff", "best_ranked", "target"]].head()

## Build The Shared Feature And Temporal Contracts

Dates stay separate from model features. The date-grouped holdout moves the nominal cutoff to the first row of its calendar date, so training dates are strictly earlier than testing dates.

In [ ]:
training_data = PreparedTrainingData(
    features=select_model_features(prepared).reset_index(drop=True),
    target=prepared["target"].to_numpy(dtype="int64"),
    dates=prepared["date"].reset_index(drop=True),
)
assert tuple(training_data.features.columns) == MODEL_FEATURE_COLUMNS

holdout = date_grouped_temporal_holdout(training_data)
X_train = training_data.features.iloc[holdout.train_indices].copy()
X_test = training_data.features.iloc[holdout.test_indices].copy()
y_train = training_data.target[holdout.train_indices].copy()
y_test = training_data.target[holdout.test_indices].copy()

train_end = training_data.dates.iloc[holdout.train_indices].max()
test_start = training_data.dates.iloc[holdout.test_indices].min()
assert train_end < test_start
{
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "train_end": train_end.date().isoformat(),
    "test_start": test_start.date().isoformat(),
}

## Compare Constant, Elo, And LightGBM In Memory

All three candidates use the same rows and shared probability evaluator. Log loss selects; accuracy, Brier score, and ROC AUC are diagnostics.

In [ ]:
prevalence = float(np.mean(y_train))
elo_model = EloBaselineClassifier().fit(X_train, y_train)
lightgbm_parameters = get_lgb_params()
lightgbm_model = build_model(lightgbm_parameters).fit(X_train, y_train)

probabilities = {
    "constant": np.full(len(y_test), prevalence),
    "elo": elo_model.predict_proba(X_test)[:, 1],
    "lightgbm": lightgbm_model.predict_proba(X_test)[:, 1],
}
candidate_metrics = {
    "constant": evaluate_probabilities(y_test, probabilities["constant"]),
    "elo": evaluate_model(elo_model, X_test, y_test),
    "lightgbm": evaluate_model(lightgbm_model, X_test, y_test),
}
candidates = candidates_with_selection_details(
    candidate_metrics, prevalence, lightgbm_parameters
)
selection = compute_selection(candidates)

pd.DataFrame(candidate_metrics).T, selection

## Small Reliability Diagnostic: Not Calibration Evidence

Binning the selected sample probabilities can reveal obvious behavior worth investigating. With only 54 holdout rows from a transformation sample, this chart is not a reliability gate, a fitted calibrator, or proof that probabilities are calibrated.

In [ ]:
selected_name = "elo" if selection["selected_model_kind"] == "elo_baseline" else "lightgbm"
reliability = pd.DataFrame(
    {"probability": probabilities[selected_name], "observed": y_test}
)
reliability["bin"] = pd.cut(
    reliability["probability"], bins=np.linspace(0.0, 1.0, 6), include_lowest=True
)
reliability = reliability.groupby("bin", observed=False).agg(
    mean_probability=("probability", "mean"),
    observed_rate=("observed", "mean"),
    rows=("observed", "size"),
)

ax = reliability.plot.scatter(
    x="mean_probability", y="observed_rate", s=60, xlim=(0, 1), ylim=(0, 1)
)
ax.plot([0, 1], [0, 1], linestyle="--", color="grey")
ax.set_title("Sample reliability diagnostic (not a calibration gate)")
plt.show()
reliability

## Return To The Non-Mutating Workflow

Use just train-check for the canonical 44,617-row boundary, just notebook-check to rerun this notebook without modifying it, and just review for the complete Docker-independent gate. Artifact regeneration is intentionally separate.